# Laboratorio: Construcción de un Sistema RAG
**Curso:** Inteligencia Artificial / Ingeniería en Sistemas  
**Tecnologías:** Python, Google Colab, FAISS, Groq (LLaMA 3), LangChain, HuggingFace Embeddings

---
## Objetivo
Construir un sistema básico de Retrieval-Augmented Generation (RAG) que combine búsqueda semántica, embeddings, bases vectoriales y un modelo de lenguaje (LLM).

## Paso 1 — Instalación de librerías

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install tiktoken
!pip install langchain-groq
!pip install langchain-huggingface
!pip install sentence-transformers

## Paso 2 — Configurar API Key (Groq)

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "TU_API_KEY_DE_GROQ_AQUI"

## Paso 3 — Base de conocimiento (10 documentos técnicos)

In [ ]:
documents = [
    "Python es un lenguaje de programación muy usado en inteligencia artificial.",
    "FAISS es una biblioteca creada por Facebook para búsqueda vectorial eficiente.",
    "Los embeddings convierten texto en vectores numéricos.",
    "RAG significa Retrieval Augmented Generation.",
    "LangChain permite construir aplicaciones con modelos de lenguaje.",
    "Kubernetes es un sistema para automatizar el despliegue de contenedores.",
    "Docker permite empaquetar aplicaciones en contenedores portables.",
    "Linux es un sistema operativo de código abierto ampliamente usado en servidores.",
    "Una red de computadoras conecta dispositivos para compartir recursos e información.",
    "Git es un sistema de control de versiones distribuido creado por Linus Torvalds."
]

print(f"Total de documentos: {len(documents)}")

## Paso 4 — Fragmentación (Chunking)

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)
docs = text_splitter.create_documents(documents)

print(f"Documentos creados: {len(docs)}")

## Paso 5 — Embeddings con HuggingFace (gratuito)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embeddings cargados correctamente")

## Paso 6 — Base de datos vectorial con FAISS

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

print("Base vectorial creada correctamente")

## Paso 7 — Consultas y generación de respuestas
### Consulta 1: ¿Qué es Kubernetes?

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")

query = "¿Qué es Kubernetes?"
results = vectorstore.similarity_search(query)

print("--- Fragmentos recuperados ---")
for r in results:
    print(r.page_content)

context = "\n".join([doc.page_content for doc in results])
prompt = f"""
Usa el siguiente contexto para responder la pregunta.

Contexto:
{context}

Pregunta:
{query}
"""

response = llm.invoke(prompt)
print("\n--- Respuesta del LLM ---")
print(response.content)

### Consulta 2: ¿Qué es Docker?

In [ ]:
query = "¿Qué es Docker?"
results = vectorstore.similarity_search(query)

print("--- Fragmentos recuperados ---")
for r in results:
    print(r.page_content)

context = "\n".join([doc.page_content for doc in results])
prompt = f"""
Usa el siguiente contexto para responder la pregunta.

Contexto:
{context}

Pregunta:
{query}
"""

response = llm.invoke(prompt)
print("\n--- Respuesta del LLM ---")
print(response.content)

### Consulta 3: ¿Qué es React? (información NO disponible en la base)

In [ ]:
query = "¿Qué es React?"
results = vectorstore.similarity_search(query)

print("--- Fragmentos recuperados ---")
for r in results:
    print(r.page_content)

context = "\n".join([doc.page_content for doc in results])
prompt = f"""
Usa el siguiente contexto para responder la pregunta.

Contexto:
{context}

Pregunta:
{query}
"""

response = llm.invoke(prompt)
print("\n--- Respuesta del LLM ---")
print(response.content)

---
## Análisis de Resultados

### 1. ¿El sistema recuperó información correcta?
Sí. En las consultas 1 y 2 (Kubernetes y Docker), el sistema recuperó correctamente los fragmentos más relevantes de la base vectorial. FAISS utilizó similitud semántica para encontrar los documentos más cercanos al significado de la pregunta, no solo por palabras clave. El LLM luego expandió esa información con una respuesta clara y estructurada.

### 2. ¿Qué ocurre si la información no está en la base?
En la consulta 3 (React), la información no existía en los documentos. FAISS devolvió los fragmentos más cercanos semánticamente (Linux, FAISS, Git, Kubernetes), que no eran relevantes para la pregunta. El LLM reconoció que el contexto no contenía información sobre React y lo indicó explícitamente, ofreciendo información general de su conocimiento propio. Esto demuestra que el modelo fue transparente sobre la limitación.

### 3. ¿El modelo alucina respuestas?
En este laboratorio el modelo no alucinó de forma problemática. Cuando la información estaba disponible, respondió con base en el contexto. Cuando no estaba disponible (React), indicó claramente que el contexto no contenía esa información. Sin embargo, en sistemas RAG reales el riesgo de alucinación existe cuando el modelo mezcla el contexto recuperado con su conocimiento interno sin distinguir claramente las fuentes.

---
## Preguntas de Reflexión

### 1. ¿Cuál es la diferencia entre un chatbot normal y un sistema RAG?
Un chatbot normal responde únicamente con el conocimiento que fue entrenado previamente en el modelo. Sus respuestas están limitadas a su fecha de corte de entrenamiento y no puede acceder a información nueva o específica de un dominio. Un sistema RAG, en cambio, primero busca información relevante en una base de conocimiento externa (vectorial), y luego usa esa información como contexto para que el LLM genere la respuesta. Esto permite que el sistema responda con datos actualizados, privados o específicos sin necesidad de reentrenar el modelo.

### 2. ¿Por qué RAG reduce las alucinaciones del modelo?
RAG reduce las alucinaciones porque ancla la respuesta del modelo a fragmentos de texto concretos recuperados de una fuente confiable. En lugar de que el LLM genere información desde su memoria interna (que puede ser imprecisa o inventada), se le proporciona un contexto específico y se le instruye a responder basándose en él. Esto limita el espacio de respuestas posibles a lo que realmente está documentado.

### 3. ¿Qué ventajas tiene usar una base vectorial?
Las bases vectoriales como FAISS permiten búsqueda semántica, es decir, encontrar documentos por significado y no solo por coincidencia exacta de palabras. Esto es más poderoso que una búsqueda tradicional porque puede recuperar información relevante aunque la pregunta esté formulada de forma diferente al documento original. Además, son altamente eficientes para manejar grandes volúmenes de datos y permiten actualizar el conocimiento del sistema simplemente agregando nuevos documentos, sin reentrenar ningún modelo.